# Exam Cheating Detection v4 — video-level decisions on v3 features

The test set consists of ~96 source videos cut into shuffled 1-s windows. Re-linking windows by frame
continuity (tools/test_link_chains.py) recovers the videos; clustering them by angle (pose) and clothing
(tools/test_subject_groups.py, side groups checked by eye) gives 6 subjects x 2 angles x 8 videos —
the same layout as every training subject (one video per behaviour and angle, 4 normal + 4 cheat).

This notebook trains the multiple-instance head on whole-video bags of the v3 window features
(V-JEPA 2 / VideoMAE / pose), GroupKFold by subject, and compares three decisions per video:
* `thr`    – cheat probability > 0.5 (needs only the re-linked videos)
* `rank`   – top-4 of the 8 videos of a (subject, angle) group are cheat
* `assign` – Hungarian assignment of the 8 videos of a group to the 8 behaviours

Outputs: `submission_pure.csv` (v3 per-window, no test structure), `submission_video.csv` (thr),
`submission_group.csv` (best of rank/assign), `cv_report.json`.

In [ ]:
import os, json, math, random, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

DEV = "cuda" if torch.cuda.is_available() else "cpu"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)


def find_dir(required, bases=("/kaggle/input", "./dataset", "./outputs")):
    for base in map(Path, bases):
        if not base.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(base):
            if all(r in filenames or r in dirnames for r in required):
                return Path(dirpath)
            dirnames[:] = [d for d in dirnames if not d.startswith(("test_seq_", "subject_"))]
    raise FileNotFoundError(required)


DATA = find_dir(["sample_submission.csv", "train"])
V2 = find_dir(["train_pose.npy", "test_pose.npy", "train_videos.csv"])
V3 = find_dir(["train_vjepa_fp16.npy", "train_windows.csv", "submission.csv"])
print("DATA", DATA, "| V2", V2, "| V3", V3, "| device", DEV)

_LOCAL = Path("outputs/structure/test_videos.json")   # embedded at build time for Kaggle (tools/build_v4.py)
STRUCTURE = json.load(open(_LOCAL)) if _LOCAL.exists() else json.loads(r'''{"chains":[[455,101,879,1301,848,823,1291,470,896,1152,1281,1260,1249,1103,1181,379,515],[632,1125,260,923,675,1235,1283,1322,1383,399,803,1351,1468,1409,1339,517],[705,1055,349,1199,554,1300,812,453,609,1384,528,148,258,1050,519],[1120,440,1370,242,328,343,692,1483,1363,1228,979,1302,134,1061,391,533],[487,86,1410,1317,1469,789,844,1059,1238,987,872,245,1407,195,28],[423,725,299,1324,345,1051,129,1119,773,827,1123,690,1507,1310,30],[978,261,209,307,347,1406,838,627,1029,263,402,17,506,1462,306,32],[522,784,834,928,196,1271,24,1018,1430,1043,796,555,1487,845,41,33],[1030,817,218,1413,99,638,72,1007,942,50,779,693,976,758,1020,37],[262,194,369,1014,1318,1492,914,1463,767,839,524,1405,1375,639,274,1024,1064],[1127,1433,137,88,650,658,119,255,1489,147,1034,1306,717,818,472,1220,1065],[253,924,404,1274,580,1026,431,370,595,300,1013,1082,1497,1266,305,42],[8,2,1262,107,977,1225,1096,611,1473,1347,667,1197,948,74,188,1503,44],[603,631,126,427,674,665,77,764,434,1505,1215,989,1381,385,1068],[621,1165,599,326,484,216,585,947,442,563,870,1504,483,311,557],[762,144,1149,771,89,747,413,254,278,664,420,1151,619,1461,108,1338,564],[520,521,1327,635,975,1293,1490,1243,1475,545,1218,1385,1182,710,815,60],[162,1136,366,1449,1458,408,819,1395,722,46,630,868,1179,610,939,715,67],[61,929,441,1189,182,481,1498,944,986,904,1397,488,607,737,573,579],[792,136,895,739,295,795,375,793,648,334,1502,538,1057,934,1453,713,1097],[149,841,887,1491,51,1094,1237,207,229,831,284,237,1299,740,971,465,588],[788,1169,57,464,409,832,380,628,733,34,1093,973,1248,1021,790,374,592],[842,578,1112,479,342,1472,1368,1456,1441,232,104,996,1186,505,83],[769,390,114,562,154,1369,1198,1357,1471,356,1156,1204,1217,1254,1110],[65,81,69,575,661,1277,678,1184,407,1343,153,474,654,292,1344,1115],[111,1486,559,507,435,35,840,1257,451,1203,1337,912,509,478,1017,433,1117],[141,1211,1188,352,1166,1467,1485,1086,124,1004,681,1223,671,959,95],[1150,688,332,701,1346,246,1253,468,864,1205,1348,527,1333,1336,624],[1284,310,847,313,449,155,570,367,862,889,657,363,1431,1106,830,1137],[495,181,652,933,534,290,787,1016,308,1426,510,110,560,846,118],[1330,1457,241,1289,169,537,761,965,963,734,1240,1308,613,826,49,190,1143],[235,358,191,782,1448,1168,1035,1227,995,604,1399,992,726,166,120],[1192,1303,264,471,361,85,738,962,601,202,52,1134,1196,323,636],[392,498,1382,583,158,372,315,946,22,117,724,314,1373,1078,445,655],[210,526,968,1162,249,1128,1124,980,1214,1295,458,629,1036,1334,187,666],[1354,376,1387,1167,1185,1474,1233,814,1070,1067,485,993,185,670],[429,439,183,835,999,1239,96,1513,270,856,53,161,201,935,677],[1434,1102,353,302,799,151,178,1145,1139,641,457,810,732,1098,997,165],[294,1429,68,763,697,1009,304,998,1341,106,230,1246,1088,179,54,679],[297,778,316,5,672,1100,79,398,1509,755,48,1038,1104,456,219,172],[750,1076,682,1425,956,333,1089,951,47,1495,285,1278,397,913,549,686],[145,58,1052,1404,516,410,957,4,64,625,389,167,309,364,382,176],[354,438,337,1403,76,546,475,443,1307,535,1427,432,23,473,586,945,1201],[577,905,103,1116,1265,1500,949,256,1514,1178,873,1440,1510,1032,883,691],[565,288,901,298,38,289,1290,866,805,90,597,248,265,205,180],[1415,82,591,1358,1488,215,1244,875,351,395,98,899,142,972,1362,1210],[643,1329,708,1231,1511,921,36,1464,286,774,477,1279,843,197],[696,1132,684,377,11,1155,0,730,1259,417,687,236,922,277,121,711],[894,581,271,84,860,1002,1372,70,238,917,1073,1273,1455,707,587,712],[1493,623,702,373,572,123,94,312,424,897,1432,199,406,622,213,384,1245],[200,454,467,186,931,886,462,703,387,12,645,1174,150,854,1361,742],[436,159,566,926,1054,112,1161,1342,221,164,1477,163,1039,482,743],[683,593,1154,550,403,1355,125,640,1108,614,529,500,541,6,1256],[1090,146,130,757,492,801,1224,511,317,885,1331,620,1144,1023,785,748],[777,1446,327,320,709,346,1356,414,783,359,1084,877,1159,558,754],[1147,45,177,1466,553,981,259,695,813,16,266,1058,168,1012,1071,251],[1484,1272,325,837,680,1075,874,231,569,208,1280,1141,1019,336,430,770],[988,822,582,1222,888,633,890,1226,1401,198,1022,291,982,768,267],[543,617,561,590,514,446,800,955,381,1445,490,1442,656,486,1027,319,1069,296,1294],[1296,234,706,1506,598,1001,1230,1160,193,1282,1091,1481,920,1208,518,272],[128,749,1175,419,1045,1083,765,1420,735,676,1379,73,1081,1496,802,786],[1183,1320,331,1482,932,211,576,1512,781,244,1371,189,494,1131,279,287],[1130,906,78,1049,532,1170,918,393,1122,1470,952,1158,729,1313,804],[718,1101,857,1325,1478,530,552,1439,1109,589,548,31,544,836,1340],[605,55,504,731,1213,602,321,574,1286,536,480,626,412,902,1353],[1193,20,1319,1232,1190,615,1451,644,668,18,1315,386,324,1328,338],[1386,1157,646,1047,1437,283,853,983,766,1015,869,491,753,1040,851],[476,634,1335,1250,257,1452,1380,281,1234,727,132,1501,138,1349,852,156,1364],[1374,1263,273,452,360,642,893,689,984,669,1421,1092,233,1321,7,344],[335,1494,884,425,9,1187,824,1053,820,1046,1326,184,1060,1056,105,858],[525,1133,1345,499,659,1236,503,43,892,322,1105,59,1140,704,808,861],[93,744,217,226,1000,1365,160,40,1111,1288,1180,109,974,1417,62,350],[1107,1118,1422,694,301,1376,1350,275,1221,496,418,1241,1113,772,943,1389],[859,794,1360,1270,925,1176,871,1416,383,1066,75,174,1207,1099,1392],[39,1252,1332,378,828,1195,113,340,539,1258,206,760,10,1298,1153,881],[362,791,1508,1229,243,1028,66,721,100,1398,339,250,122,268,1393],[1164,1114,318,240,1079,960,1206,970,720,1412,1085,224,698,1359,880],[1212,157,461,14,1423,1146,269,612,502,1121,1480,204,616,401,1408],[444,1148,759,850,71,1269,1171,116,341,910,1454,649,653,1037,1411],[915,1323,662,596,1138,600,663,863,1267,833,469,1063,567,1460,1414],[223,1010,25,908,1126,416,152,876,825,133,556,618,437,1402,1418],[1142,637,135,608,212,459,1074,282,175,1006,806,1311,422,357,365,396],[1033,1077,1316,1219,1285,466,551,1276,673,919,891,1264,547,930,411],[1209,497,1072,1242,1436,985,203,958,745,1465,967,1,966,448,568,927],[1095,1352,63,1268,1080,1428,1400,450,1216,1367,1443,882,938,1005,1444],[1194,1041,214,961,716,508,293,1304,228,463,1396,1435,1312,21,728,1447],[1378,415,1163,1309,1438,1292,1261,807,15,1390,1172,907,584,512,280,1048,247,940],[27,821,139,80,1459,606,405,220,1177,1314,127,746,776,829,941],[714,865,400,493,542,900,647,775,192,1044,798,29,489,330,428],[140,92,501,91,131,447,1087,867,143,719,916,276,660,394,953],[969,348,13,1025,855,523,388,1200,936,723,736,115,991,1297,1419,368,1476],[954,1202,87,1062,950,371,849,1391,594,911,1011,170,909,1450,1251,1479],[898,1042,1173,239,903,97,222,227,3,1191,1287,1394,651,225,460],[780,1366,513,751,1388,252,426,171,756,1255,811,741,685,173,994],[303,1377,1499,1031,102,571,937,1135,1247,1424,531,1275,355,699,1305,1008],[26,56,700,540,964,816,1003,809,797,990,1129,421,329,19,752,878]],"group":[6,4,4,11,9,6,11,5,0,5,10,1,7,2,5,10,3,8,0,6,9,8,11,1,11,7,6,9,7,11,7,0,3,9,11,11,1,8,7,9,10,5,5,9,2,9,1,6,3,8,11,8,4,10,2,8,5,0,7,10,3,3,3,4,4,1,4,1,6,7,7,5,3,9,2,2,4,2,1,0,10,2,1,0,2,0,10,6,4,5,6,10,3,0,8,8],"angle":["side","front","front","side","side","side","side","front","front","front","side","front","side","front","front","side","front","side","front","side","side","side","side","front","side","side","side","side","side","side","side","front","front","side","side","side","front","side","side","side","side","front","front","side","front","side","front","side","front","side","side","side","front","side","front","side","front","front","side","side","front","front","front","front","front","front","front","front","side","side","side","front","front","side","front","front","front","front","front","front","side","front","front","front","front","front","side","side","front","front","side","side","front","front","side","side"],"note":"96 test videos = re-linked chains (63+74 merged); group = (angle, subject) clusters, side 6/8 hand-corrected"}''')
test_ids = pd.read_csv(DATA / "sample_submission.csv").id.tolist()
chains, groups = STRUCTURE["chains"], np.array(STRUCTURE["group"])
assert sorted(sum(chains, [])) == list(range(len(test_ids)))
print(len(chains), "test videos;", np.bincount(groups).tolist(), "videos per (subject, angle) group")

T = 30
vdf = pd.read_csv(V2 / "train_videos.csv")
wdf = pd.read_csv(V3 / "train_windows.csv")

## Features (same as v3)

In [ ]:
def featurize(W):
    B, Tn, _ = W.shape
    kp = W[..., :51].reshape(B, Tn, 17, 3)
    xy, cf = kp[..., :2], kp[..., 2]
    box, bc = W[..., 51:55], W[..., 55:]
    valid = (bc > 0).float()
    nv = valid.sum(1).clamp(min=1)
    sh = (xy[:, :, 5] + xy[:, :, 6]) / 2
    ref = (sh * valid).sum(1) / nv
    bh = ((box[..., 3] - box[..., 1]) * valid[..., 0]).sum(1) / nv[:, 0]
    scale = bh.clamp(min=0.05)[:, None, None, None]
    rel = (xy - ref[:, None, None]) / scale * valid[..., None]
    vel = torch.diff(rel, dim=1, prepend=rel[:, :1])
    return torch.cat([xy.flatten(2), cf, rel.flatten(2), vel.flatten(2) * 10, box, bc], -1)


def window_stats(f):
    return torch.cat([f.mean(1), f.std(1), f.amin(1), f.amax(1), (f[:, 1:] - f[:, :-1]).abs().mean(1)], 1)


@torch.no_grad()
def pose_stats(Wall, bs=4096):
    Wall = torch.from_numpy(np.ascontiguousarray(Wall)).float()
    return torch.cat([window_stats(featurize(Wall[i:i + bs].to(DEV))).cpu() for i in range(0, len(Wall), bs)]).numpy()


train_pose, test_pose = np.load(V2 / "train_pose.npy"), np.load(V2 / "test_pose.npy")
P_TR = pose_stats(train_pose[wdf.gstart.values[:, None] + np.arange(T)])
P_TE = pose_stats(test_pose)
FE = {k: (np.load(V3 / f"train_{k}_fp16.npy"), np.load(V3 / f"test_{k}_fp16.npy")) for k in ["vjepa", "vmae"]}
HID = {"vjepa": 1024, "vmae": 768}


def block(name, which):
    if name == "pose":
        return P_TR, P_TE
    d = HID[name]
    nl = FE[name][0].shape[1] - 6 * d
    sl = dict(pooled=np.arange(0, d), grid=np.arange(2 * d, 6 * d), logits=np.arange(6 * d, 6 * d + nl))
    cols = np.concatenate([sl[w] for w in which])
    return FE[name][0][:, cols].astype(np.float32), FE[name][1][:, cols].astype(np.float32)


CONFIGS = {"vjepa": [("vjepa", ["pooled", "logits"])], "pose": [("pose", None)],
           "vjepa_grid": [("vjepa", ["pooled", "grid", "logits"])], "vmae": [("vmae", ["pooled", "logits"])],
           "all": [("vjepa", ["pooled", "logits"]), ("vmae", ["pooled", "logits"]), ("pose", None)]}

## Multiple-instance head on whole-video bags

In [ ]:
class MIL(nn.Module):
    def __init__(self, din, d=128, p=0.4):
        super().__init__()
        self.enc = nn.Sequential(nn.Dropout(0.25), nn.Linear(din, d), nn.GELU(), nn.Dropout(p),
                                 nn.Linear(d, d), nn.GELU())
        self.att = nn.Sequential(nn.Linear(d, 64), nn.Tanh(), nn.Linear(64, 1))
        self.drop = nn.Dropout(p)
        self.out_bin, self.out_beh = nn.Linear(2 * d, 1), nn.Linear(2 * d, 8)

    def forward(self, x, mask):
        h = self.enc(x)
        a = self.att(h).squeeze(-1).masked_fill(~mask, -1e4).softmax(1)
        z = torch.cat([(a[..., None] * h).sum(1), h.masked_fill(~mask[..., None], -1e4).amax(1)], 1)
        z = self.drop(z)
        return self.out_bin(z).squeeze(1), self.out_beh(z)


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


V_LAB, V_BEH = vdf.label.values, vdf.beh_id.values
VID_ROWS = {v: np.where(wdf.video.values == v)[0] for v in range(len(vdf))}
VID_EVAL = {v: r[wdf.off.values[r] % T == 0] for v, r in VID_ROWS.items()}   # 1-s windows aligned like the test
SEEDS, EPOCHS = ([42], 1) if os.environ.get("SMOKE") == "1" else ([42, 7, 2026], 60)


def pad(bags, X):
    L = max(len(b) for b in bags)
    idx = np.zeros((len(bags), L), np.int64); mask = np.zeros((len(bags), L), bool)
    for i, b in enumerate(bags):
        idx[i, :len(b)] = b; mask[i, :len(b)] = True
    return X[torch.from_numpy(idx).to(DEV)], torch.from_numpy(mask).to(DEV)


def train_mil(X, videos, seed):
    seed_all(seed)
    model = MIL(X.shape[1]).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
    steps = EPOCHS * math.ceil(len(videos) * 8 / 64)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=steps, pct_start=0.1)
    for _ in range(EPOCHS):
        model.train()
        order = np.random.permutation(np.repeat(videos, 8))
        for i in range(0, len(order), 64):
            L = random.choice([1, 2, 4, 8, 12, 15, 30])
            bags = []
            for v in order[i:i + 64]:
                seq = VID_ROWS[v][random.randint(0, 1)::2]
                l = min(L, len(seq)); s0 = random.randint(0, len(seq) - l)
                bags.append(seq[s0:s0 + l])
            x, m = pad(bags, X)
            lb, lh = model(x + 0.1 * torch.randn_like(x), m)
            vv = order[i:i + 64]
            loss = F.binary_cross_entropy_with_logits(lb, torch.as_tensor(V_LAB[vv], device=DEV).float() * 0.9 + 0.05) \
                + 0.5 * F.cross_entropy(lh, torch.as_tensor(V_BEH[vv], device=DEV), label_smoothing=0.1)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
    return model.eval()


@torch.no_grad()
def predict(model, bags, X):
    x, m = pad(bags, X)
    lb, lh = model(x, m)
    return torch.sigmoid(lb).cpu().numpy(), lh.softmax(1).cpu().numpy()


folds = list(GroupKFold(n_splits=5).split(vdf, groups=vdf.subject))
res_va, res_te = {}, {}
t0 = time.time()
for cname, parts in CONFIGS.items():
    bl = [block(n, w) for n, w in parts]
    Xtr_np, Xte_np = np.concatenate([b[0] for b in bl], 1), np.concatenate([b[1] for b in bl], 1)
    pb, ph = np.zeros(len(vdf)), np.zeros((len(vdf), 8))
    tb, th = np.zeros(len(chains)), np.zeros((len(chains), 8))
    for fold, (tr, va) in enumerate(folds):
        rows = np.concatenate([VID_ROWS[v] for v in tr])
        mu, sd = Xtr_np[rows].mean(0), Xtr_np[rows].std(0) + 1e-3
        Xtr = torch.from_numpy(np.clip((Xtr_np - mu) / sd, -6, 6)).float().to(DEV)
        Xte = torch.from_numpy(np.clip((Xte_np - mu) / sd, -6, 6)).float().to(DEV)
        for seed in SEEDS:
            m = train_mil(Xtr, tr, seed + 100 * fold)
            b, h = predict(m, [VID_EVAL[v] for v in va], Xtr)
            pb[va] += b / len(SEEDS); ph[va] += h / len(SEEDS)
            b, h = predict(m, chains, Xte)
            tb += b / (len(SEEDS) * len(folds)); th += h / (len(SEEDS) * len(folds))
    res_va[cname], res_te[cname] = (pb, ph), (tb, th)
    print(f"{cname:11s} video auc bin={roc_auc_score(V_LAB, pb):.4f} beh-cheat={roc_auc_score(V_LAB, ph[:, 4:].sum(1)):.4f} "
          f"behaviour acc={(ph.argmax(1) == V_BEH).mean():.4f} {time.time() - t0:.0f}s", flush=True)

## Decisions, CV report (window-weighted accuracy = accuracy on 1-s test-like windows)

In [ ]:
def assign(beh_p):
    cost = -np.log(np.clip(beh_p, 1e-6, 1))
    cost = np.concatenate([cost, cost + 3.0], 1)          # a behaviour may be reused only at a high cost
    r, c = linear_sum_assignment(cost)
    out = np.zeros(len(beh_p), int); out[r] = c % 8
    return out


def decide(pbin, pbeh, grp):
    p = 0.5 * pbin + 0.5 * pbeh[:, 4:].sum(1)
    rank, beh = np.zeros(len(p), int), np.zeros(len(p), int)
    for g in np.unique(grp):
        idx = np.where(grp == g)[0]
        rank[idx[np.argsort(-p[idx])[:int(round(len(idx) / 2))]]] = 1
        beh[idx] = assign(pbeh[idx])
    return p, {"thr": (p > 0.5).astype(int), "rank": rank, "assign": (beh >= 4).astype(int), "beh": beh}


w_vid = np.array([len(VID_EVAL[v]) for v in range(len(vdf))])
g_tr = pd.factorize(vdf.subject.astype(str) + "_" + vdf.angle)[0]
combos = {c: [c] for c in CONFIGS}
combos.update({"vjepa+pose": ["vjepa", "pose"], "vjepa+pose+grid": ["vjepa", "pose", "vjepa_grid"],
               "vjepa+vmae+pose": ["vjepa", "vmae", "pose"], "all5": list(CONFIGS)})
report = {}
for name, cs in combos.items():
    pb = np.mean([res_va[c][0] for c in cs], 0); ph = np.mean([res_va[c][1] for c in cs], 0)
    p, d = decide(pb, ph, g_tr)
    report[name] = {k: round(float(np.average(d[k] == V_LAB, weights=w_vid)), 4) for k in ["thr", "rank", "assign"]}
    report[name].update(auc=round(float(roc_auc_score(V_LAB, p)), 4), behaviour_acc=round(float((d["beh"] == V_BEH).mean()), 4))
    print(f"{name:16s}", report[name])

best_thr = max(report, key=lambda k: report[k]["thr"])
best_grp = max(((k, m) for k in report for m in ["rank", "assign"]), key=lambda km: report[km[0]][km[1]])
print("best video-only:", best_thr, report[best_thr]["thr"], "| best group:", best_grp, report[best_grp[0]][best_grp[1]])

pb = np.mean([res_va[c][0] for c in combos[best_grp[0]]], 0); ph = np.mean([res_va[c][1] for c in combos[best_grp[0]]], 0)
_, d = decide(pb, ph, g_tr)
cm = pd.crosstab(vdf.behavior, pd.Series(np.array(["Writing_Answers", "Thinking", "Looking_Around", "Checking_Smartwatch",
                 "Looking_Under_the_Desk", "Reading_Hidden_Notes", "Repeated_Peeking", "Using_Smartwatch"])[d["beh"]], name="assigned"))
print(cm)

## Submissions

In [ ]:
def to_windows(video_labels):
    lab = np.zeros(len(test_ids), int)
    for ch, y in zip(chains, video_labels):
        lab[ch] = y
    return lab


sub = pd.read_csv(DATA / "sample_submission.csv")
shutil.copy(V3 / "submission.csv", WORK / "submission_pure.csv")

pb = np.mean([res_te[c][0] for c in combos[best_thr]], 0); ph = np.mean([res_te[c][1] for c in combos[best_thr]], 0)
_, d = decide(pb, ph, groups)
sub.assign(label=to_windows(d["thr"])).to_csv(WORK / "submission_video.csv", index=False)

pb = np.mean([res_te[c][0] for c in combos[best_grp[0]]], 0); ph = np.mean([res_te[c][1] for c in combos[best_grp[0]]], 0)
p_te, d = decide(pb, ph, groups)
sub.assign(label=to_windows(d[best_grp[1]])).to_csv(WORK / "submission_group.csv", index=False)
pd.DataFrame({"video": range(len(chains)), "group": groups, "n_windows": [len(c) for c in chains], "prob": p_te,
              "behaviour": d["beh"], "thr": d["thr"], "rank": d["rank"], "assign": d["assign"]}).to_csv(WORK / "test_videos.csv", index=False)

for f in ["submission_pure.csv", "submission_video.csv", "submission_group.csv"]:
    s = pd.read_csv(WORK / f)
    print(f, s.label.value_counts().to_dict())
json.dump({"cv": report, "best_video_only": [best_thr, report[best_thr]["thr"]],
           "best_group": [*best_grp, report[best_grp[0]][best_grp[1]]]}, open(WORK / "cv_report.json", "w"), indent=1)